## Construyo .csv's de los listados de partidos 2018-2022

In [41]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

statsbomb_dir = BASE_DIR / "data" / "raw" / "statsbomb"
processed_dir = BASE_DIR / "data" / "processed"

matches_2018_path = statsbomb_dir / "data" / "matches" / "43" / "3.json"
matches_2022_path = statsbomb_dir / "data" / "matches" / "43" / "106.json"

matches_2018_path.exists(), matches_2022_path.exists()

(True, True)

In [42]:
with open(matches_2018_path, "r", encoding="utf-8") as f:
    matches_2018_json = json.load(f)

with open(matches_2022_path, "r", encoding="utf-8") as f:
    matches_2022_json = json.load(f)

len(matches_2018_json), len(matches_2022_json)

(64, 64)

In [43]:
for key, value in matches_2022_json[0].items():
    print(key)

match_id
match_date
kick_off
competition
season
home_team
away_team
home_score
away_score
match_status
match_status_360
last_updated
last_updated_360
metadata
match_week
competition_stage
stadium
referee


In [44]:
def build_matches_index(matches_json, year):
    rows = []

    for match in matches_json:
        rows.append({
            "match_id": match["match_id"],
            "match_date": match["match_date"],
            "kick_off": match["kick_off"],
            "year": year,
            "home_team": match["home_team"]["home_team_name"],
            "away_team": match["away_team"]["away_team_name"],
            "home_score": match["home_score"],
            "away_score": match["away_score"],
            "competition_stage": match["competition_stage"]["name"],
        })

    return pd.DataFrame(rows)

In [45]:
matches_index_2018 = build_matches_index(matches_2018_json, 2018)
matches_index_2022 = build_matches_index(matches_2022_json, 2022)

matches_index_v01 = pd.concat(
    [matches_index_2018, matches_index_2022],
    ignore_index=True
)

matches_index_v01["match_date"] = pd.to_datetime(matches_index_v01["match_date"])
matches_index_v01 = matches_index_v01.sort_values(
    ["year", "match_date", "kick_off"]
).reset_index(drop=True)

matches_index_v01.shape

(128, 9)

In [46]:
matches_index_v01.head()

,match_id,match_date,kick_off,year,home_team,away_team,home_score,away_score,competition_stage
0,7525,2018-06-14,15:00:00.000,2018,Russia,Saudi Arabia,5,0,Group Stage
1,7578,2018-06-15,12:00:00.000,2018,Egypt,Uruguay,0,1,Group Stage
2,7577,2018-06-15,15:00:00.000,2018,Morocco,Iran,0,1,Group Stage
3,7576,2018-06-15,18:00:00.000,2018,Portugal,Spain,3,3,Group Stage
4,7530,2018-06-16,10:00:00.000,2018,France,Australia,2,1,Group Stage


In [47]:
matches_index_v01.tail()

,match_id,match_date,kick_off,year,home_team,away_team,home_score,away_score,competition_stage
123,3869354,2022-12-10,21:00:00.000,2022,England,France,1,2,Quarter-finals
124,3869519,2022-12-13,21:00:00.000,2022,Argentina,Croatia,3,0,Semi-finals
125,3869552,2022-12-14,21:00:00.000,2022,France,Morocco,2,0,Semi-finals
126,3869684,2022-12-17,17:00:00.000,2022,Croatia,Morocco,2,1,3rd Place Final
127,3869685,2022-12-18,17:00:00.000,2022,Argentina,France,3,3,Final


In [48]:
matches_index_v01.to_csv(
    processed_dir / "matches_index_v01.csv",
    index=False
)

# Construyo  pre match vector

In [49]:
team_vector = pd.read_csv(processed_dir / "team_vector_v01.csv")
matches_index = pd.read_csv(processed_dir / "matches_index_v01.csv")

matches_index["match_date"] = pd.to_datetime(matches_index["match_date"])

In [50]:
team_vector = team_vector.merge(
    matches_index[["match_id", "match_date", "year", "competition_stage"]],
    on="match_id",
    how="left"
)

team_vector = team_vector.sort_values(
    ["year", "team_name", "match_date"]
).reset_index(drop=True)

team_vector.head()

,match_id,team_name,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,Dispossessed,Dribble,Dribbled Past,...,Player On,Pressure,Shot,Offside,Shield,Bad Behaviour,50/50,match_date,year,competition_stage
0,7531,Argentina,755,54,13,698,10,18,15,6,...,0.0,84,28,0.0,0.0,0.0,3.0,2018-06-16,2018,Group Stage
1,7545,Argentina,521,48,19,455,7,7,18,16,...,1.0,236,10,0.0,1.0,1.0,2.0,2018-06-21,2018,Group Stage
2,7564,Argentina,577,52,10,471,13,5,25,18,...,0.0,134,8,0.0,0.0,1.0,2.0,2018-06-26,2018,Group Stage
3,7580,Argentina,526,42,12,491,5,15,22,11,...,0.0,180,12,0.0,0.0,2.0,1.0,2018-06-30,2018,Round of 16
4,7530,Australia,415,48,16,345,19,13,5,14,...,0.0,113,5,0.0,0.0,0.0,1.0,2018-06-16,2018,Group Stage


In [51]:
ID_COLS = [
    "match_id",
    "team_name",
    "match_date",
    "year",
    "competition_stage",
]

feature_cols = [
    col for col in team_vector.columns
    if col not in ID_COLS
]

team_pre_match_vector_v01 = team_vector.copy()

team_pre_match_vector_v01[feature_cols] = (
    team_vector
    .groupby(["year", "team_name"])[feature_cols]
    .transform(lambda x: x.shift().expanding().mean())
)

In [52]:
team_pre_match_vector_v01[feature_cols] = team_pre_match_vector_v01[feature_cols].fillna(0)

In [53]:
team_pre_match_vector_v01.shape

(256, 30)

In [54]:
team_pre_match_vector_v01.head()

,match_id,team_name,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,Dispossessed,Dribble,Dribbled Past,...,Player On,Pressure,Shot,Offside,Shield,Bad Behaviour,50/50,match_date,year,competition_stage
0,7531,Argentina,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,2018-06-16,2018,Group Stage
1,7545,Argentina,755.000000,54.000000,13.0,698.000000,10.0,18.0,15.000000,6.000000,...,0.000000,84.000000,28.000000,0.0,0.000000,0.000000,3.000000,2018-06-21,2018,Group Stage
2,7564,Argentina,638.000000,51.000000,16.0,576.500000,8.5,12.5,16.500000,11.000000,...,0.500000,160.000000,19.000000,0.0,0.500000,0.500000,2.500000,2018-06-26,2018,Group Stage
3,7580,Argentina,617.666667,51.333333,14.0,541.333333,10.0,10.0,19.333333,13.333333,...,0.333333,151.333333,15.333333,0.0,0.333333,0.666667,2.333333,2018-06-30,2018,Round of 16
4,7530,Australia,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,2018-06-16,2018,Group Stage


In [55]:
team_pre_match_vector_v01.to_csv(
    processed_dir / "team_pre_match_vector_v01.csv",
    index=False
)

In [56]:
team_pre_match = pd.read_csv(
    processed_dir / "team_pre_match_vector_v01.csv"
)

matches_index = pd.read_csv(
    processed_dir / "matches_index_v01.csv"
)

team_pre_match.shape, matches_index.shape

((256, 30), (128, 9))

In [58]:
team_pre_match.columns.tolist()

['match_id',
 'team_name',
 'Ball Receipt*',
 'Ball Recovery',
 'Block',
 'Carry',
 'Clearance',
 'Dispossessed',
 'Dribble',
 'Dribbled Past',
 'Duel',
 'Error',
 'Foul Committed',
 'Foul Won',
 'Goal Keeper',
 'Interception',
 'Miscontrol',
 'Own Goal Against',
 'Pass',
 'Player Off',
 'Player On',
 'Pressure',
 'Shot',
 'Offside',
 'Shield',
 'Bad Behaviour',
 '50/50',
 'match_date',
 'year',
 'competition_stage']

# Comienzo Match Vector

Tomo la tabla de partidos y le pego la información histórica de ambos equipos.

In [57]:
team_pre_features = team_pre_match.drop(
    columns=["match_date", "year", "competition_stage"],
    errors="ignore"
)

home_pre = team_pre_features.add_prefix("home_")
away_pre = team_pre_features.add_prefix("away_")

In [59]:
match_vector_v01 = matches_index.copy()

match_vector_v01 = match_vector_v01.merge(
    home_pre,
    left_on=["match_id", "home_team"],
    right_on=["home_match_id", "home_team_name"],
    how="left"
)

match_vector_v01 = match_vector_v01.merge(
    away_pre,
    left_on=["match_id", "away_team"],
    right_on=["away_match_id", "away_team_name"],
    how="left"
)

Limpio duplicados

In [60]:
match_vector_v01 = match_vector_v01.drop(
    columns=["home_match_id", "home_team_name", "away_match_id", "away_team_name"],
    errors="ignore"
)

Genero valor binario de target para los resultados de los partidos

In [61]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return 1
    elif row["home_score"] < row["away_score"]:
        return -1
    else:
        return 0

match_vector_v01["target"] = match_vector_v01.apply(get_result, axis=1)

Checks

In [68]:
print(match_vector_v01.shape,"\n")
print(match_vector_v01[["home_team", "away_team", "home_score", "away_score", "target"]].head(),"\n")

print("             NaNs")
print(match_vector_v01.isna().sum().sort_values(ascending=False).head(10))

(128, 60) 

  home_team     away_team  home_score  away_score  target
0    Russia  Saudi Arabia           5           0       1
1     Egypt       Uruguay           0           1      -1
2   Morocco          Iran           0           1      -1
3  Portugal         Spain           3           3       0
4    France     Australia           2           1       1 

             NaNs
match_id              0
match_date            0
kick_off              0
year                  0
home_team             0
away_team             0
home_score            0
away_score            0
competition_stage     0
home_Ball Receipt*    0
dtype: int64


In [69]:
match_vector_v01.to_csv(
    processed_dir / "match_vector_v01.csv",
    index=False
)

### Auditoría

In [73]:
print(match_vector_v01.shape)

(128, 60)


In [74]:
match_vector_v01.info()

<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 60 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   match_id               128 non-null    int64  
 1   match_date             128 non-null    str    
 2   kick_off               128 non-null    str    
 3   year                   128 non-null    int64  
 4   home_team              128 non-null    str    
 5   away_team              128 non-null    str    
 6   home_score             128 non-null    int64  
 7   away_score             128 non-null    int64  
 8   competition_stage      128 non-null    str    
 9   home_Ball Receipt*     128 non-null    float64
 10  home_Ball Recovery     128 non-null    float64
 11  home_Block             128 non-null    float64
 12  home_Carry             128 non-null    float64
 13  home_Clearance         128 non-null    float64
 14  home_Dispossessed      128 non-null    float64
 15  home_Dribble     

In [75]:
match_vector_v01["target"].value_counts()

target
 1    55
-1    45
 0    28
Name: count, dtype: int64